In [ ]:
from pathlib import Path
import platform
import shutil

WORK = Path("/content/rk3576_yolo_dataset")
WORK.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "keyboard",
    "mouse",
    "bottle",
    "cup",
    "person",
]

IMAGE_SIZE = 128

print("Dataset preparation environment")
print("Python:", platform.python_version())
print("Workspace:", WORK)
print("Target image size:", IMAGE_SIZE)
print("Classes:")

for class_id, name in enumerate(CLASSES):
    print(f"  {class_id}: {name}")

print("\nAvailable disk space:")
print(shutil.disk_usage("/content"))

Dataset preparation environment
Python: 3.13.15
Workspace: /content/rk3576_yolo_dataset
Target image size: 128
Classes:
  0: keyboard
  1: mouse
  2: bottle
  3: cup
  4: person

Available disk space:
usage(total=120942624768, used=45469749248, free=75456098304)


In [ ]:
import json
import urllib.request
import ssl
import zipfile
from pathlib import Path

WORK = Path("/content/rk3576_yolo_dataset")
ANN_DIR = WORK / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

url = "https://images.cocodataset.org/annotations/annotations_trainval2017.zip"
archive = WORK / "annotations_trainval2017.zip"

# Download only if not already present
if not archive.exists():
    print("Downloading COCO annotations (~241 MB)...")
    # Create an unverified SSL context to bypass the certificate verification error
    ssl_context = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=ssl_context) as response, open(archive, 'wb') as out_file:
        shutil_len = shutil = None  # Use basic chunked write
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out_file.write(chunk)

print("Extracting the two detection annotation files...")
with zipfile.ZipFile(archive) as z:
    for name in (
        "annotations/instances_train2017.json",
        "annotations/instances_val2017.json",
    ):
        target = ANN_DIR / Path(name).name
        if not target.exists():
            with z.open(name) as src, open(target, "wb") as dst:
                dst.write(src.read())

print("Counting target classes...")

TARGETS = ["keyboard", "mouse", "bottle", "cup", "person"]

for split in ["train2017", "val2017"]:
    path = ANN_DIR / f"instances_{split}.json"
    with open(path, "r") as f:
        data = json.load(f)

    category_ids = {
        cat["id"]: cat["name"]
        for cat in data["categories"]
        if cat["name"] in TARGETS
    }

    counts = {name: 0 for name in TARGETS}
    image_ids = {name: set() for name in TARGETS}

    for ann in data["annotations"]:
        name = category_ids.get(ann["category_id"])
        if name is not None and not ann.get("iscrowd", 0):
            counts[name] += 1
            image_ids[name].add(ann["image_id"])

    print(f"\n{split}:")
    for name in TARGETS:
        print(
            f"  {name:10s}: "
            f"{counts[name]:6d} instances, "
            f"{len(image_ids[name]):6d} images"
        )

print("\nAnnotation preparation complete.")
print("Workspace:", WORK)

Extracting the two detection annotation files...
Counting target classes...

train2017:
  keyboard  :   2854 instances,   2115 images
  mouse     :   2261 instances,   1876 images
  bottle    :  24070 instances,   8501 images
  cup       :  20574 instances,   9189 images
  person    : 257253 instances,  64115 images

val2017:
  keyboard  :    153 instances,    106 images
  mouse     :    106 instances,     88 images
  bottle    :   1013 instances,    379 images
  cup       :    895 instances,    390 images
  person    :  10777 instances,   2693 images

Annotation preparation complete.
Workspace: /content/rk3576_yolo_dataset


In [ ]:

import json
import random
from collections import Counter, defaultdict
from pathlib import Path

WORK = Path("/content/rk3576_yolo_dataset")
ANN_DIR = WORK / "annotations"

TARGETS = ["keyboard", "mouse", "bottle", "cup", "person"]
CLASS_TO_ID = {name: i for i, name in enumerate(TARGETS)}

LIMITS = {
    "train2017": 500,
    "val2017": 100,
}

SEED = 3576
rng = random.Random(SEED)

def select_images(split, limit):
    path = ANN_DIR / f"instances_{split}.json"
    with open(path, "r") as f:
        data = json.load(f)

    cat_names = {
        cat["id"]: cat["name"]
        for cat in data["categories"]
        if cat["name"] in TARGETS
    }

    images = {im["id"]: im for im in data["images"]}
    image_classes = defaultdict(set)
    image_instances = defaultdict(Counter)

    for ann in data["annotations"]:
        name = cat_names.get(ann["category_id"])
        if name is None or ann.get("iscrowd", 0):
            continue

        # Ignore invalid or zero-area boxes
        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue

        image_id = ann["image_id"]
        image_classes[image_id].add(name)
        image_instances[image_id][name] += 1

    candidates = list(image_classes.keys())
    rng.shuffle(candidates)

    # Greedy class-balanced selection.
    # Images containing several underrepresented classes are favored.
    selected = []
    selected_set = set()
    selected_class_counts = Counter()

    while len(selected) < limit:
        best_id = None
        best_score = -1

        for image_id in candidates:
            if image_id in selected_set:
                continue

            names = image_classes[image_id]
            score = sum(
                1.0 / (1 + selected_class_counts[name])
                for name in names
            )

            if score > best_score:
                best_score = score
                best_id = image_id

        if best_id is None:
            break

        selected.append(best_id)
        selected_set.add(best_id)

        for name in image_classes[best_id]:
            selected_class_counts[name] += 1

    selected_images = [images[i] for i in selected]
    selected_counts = Counter()
    instance_counts = Counter()

    for image_id in selected:
        for name in image_classes[image_id]:
            selected_counts[name] += 1
        instance_counts.update(image_instances[image_id])

    result = {
        "split": split,
        "image_ids": selected,
        "images": selected_images,
        "image_class_counts": dict(selected_counts),
        "instance_counts": dict(instance_counts),
        "image_classes": {
            str(i): sorted(image_classes[i])
            for i in selected
        },
    }

    return result

selections = {}

for split, limit in LIMITS.items():
    result = select_images(split, limit)
    selections[split] = result

    print(f"\n=== {split} ===")
    print("Selected images:", len(result["images"]))
    print("Images containing each class:")
    for name in TARGETS:
        print(f"  {name:10s}: {result['image_class_counts'].get(name, 0)}")
    print("Annotated instances:")
    for name in TARGETS:
        print(f"  {name:10s}: {result['instance_counts'].get(name, 0)}")

selection_path = WORK / "selected_images.json"
with open(selection_path, "w") as f:
    json.dump(selections, f, indent=2)

print("\nSaved image selection to:", selection_path)



=== train2017 ===
Selected images: 500
Images containing each class:
  keyboard  : 344
  mouse     : 344
  bottle    : 344
  cup       : 345
  person    : 345
Annotated instances:
  keyboard  : 500
  mouse     : 424
  bottle    : 735
  cup       : 684
  person    : 1031

=== val2017 ===
Selected images: 100
Images containing each class:
  keyboard  : 47
  mouse     : 47
  bottle    : 66
  cup       : 72
  person    : 72
Annotated instances:
  keyboard  : 85
  mouse     : 60
  bottle    : 197
  cup       : 183
  person    : 256

Saved image selection to: /content/rk3576_yolo_dataset/selected_images.json


In [ ]:
import json
import urllib.request
import ssl
import concurrent.futures
from pathlib import Path
from collections import Counter
from PIL import Image
from io import BytesIO

WORK = Path("/content/rk3576_yolo_dataset")
ANN_DIR = WORK / "annotations"
OUT = WORK / "coco5_yolo"
IMAGE_SIZE = 128

CLASSES = ["keyboard", "mouse", "bottle", "cup", "person"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

for split in ["train2017", "val2017"]:
    short = "train" if split == "train2017" else "val"
    (OUT / "images" / short).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / short).mkdir(parents=True, exist_ok=True)

with open(WORK / "selected_images.json") as f:
    selections = json.load(f)

def prepare_split(split):
    short = "train" if split == "train2017" else "val"

    with open(ANN_DIR / f"instances_{split}.json") as f:
        data = json.load(f)

    selected = {
        int(image["id"]): image
        for image in selections[split]["images"]
    }

    cat_id_to_class = {
        cat["id"]: CLASS_TO_ID[cat["name"]]
        for cat in data["categories"]
        if cat["name"] in CLASS_TO_ID
    }

    annotations = {}
    for ann in data["annotations"]:
        image_id = ann["image_id"]
        if image_id not in selected:
            continue
        if ann["category_id"] not in cat_id_to_class:
            continue
        if ann.get("iscrowd", 0):
            continue

        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue

        annotations.setdefault(image_id, []).append(
            (cat_id_to_class[ann["category_id"]], x, y, w, h)
        )

    # Create an unverified SSL context to bypass verification errors
    ssl_context = ssl._create_unverified_context()

    def download_one(item):
        image_id, meta = item
        filename = meta["file_name"]
        image_path = OUT / "images" / short / filename
        label_path = OUT / "labels" / short / (Path(filename).stem + ".txt")

        try:
            url = f"https://images.cocodataset.org/{split}/{filename}"
            req = urllib.request.Request(
                url, headers={"User-Agent": "RK3576-Dataset-Prep/1.0"}
            )
            with urllib.request.urlopen(req, timeout=30, context=ssl_context) as response:
                raw = response.read()

            with Image.open(BytesIO(raw)) as im:
                im = im.convert("RGB")
                width, height = im.size

                # Direct square resize. Normalized box coordinates remain valid.
                im = im.resize(
                    (IMAGE_SIZE, IMAGE_SIZE),
                    Image.Resampling.BILINEAR,
                )
                im.save(image_path, format="JPEG", quality=88, optimize=True)

            lines = []
            for class_id, x, y, w, h in annotations.get(image_id, []):
                # Clip box to the original image boundaries.
                x1 = max(0.0, min(float(width), x))
                y1 = max(0.0, min(float(height), y))
                x2 = max(0.0, min(float(width), x + w))
                y2 = max(0.0, min(float(height), y + h))

                bw = x2 - x1
                bh = y2 - y1
                if bw <= 0 or bh <= 0:
                    continue

                xc = ((x1 + x2) / 2) / width
                yc = ((y1 + y2) / 2) / height
                nw = bw / width
                nh = bh / height

                lines.append(
                    f"{class_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}"
                )

            if not lines:
                image_path.unlink(missing_ok=True)
                return (False, filename, "no valid boxes")

            label_path.write_text("\n".join(lines) + "\n")
            return (True, filename, len(lines))

        except Exception as e:
            image_path.unlink(missing_ok=True)
            label_path.unlink(missing_ok=True)
            return (False, filename, str(e)[:150])

    items = list(selected.items())
    results = []

    print(f"\nPreparing {short}: {len(items)} selected images")

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for result in pool.map(download_one, items):
            results.append(result)

    successful = [r for r in results if r[0]]
    failed = [r for r in results if not r[0]]

    print("Successful images:", len(successful))
    print("Failed images:", len(failed))

    if failed:
        print("First failures:")
        for item in failed[:5]:
            print(" ", item[1], "-", item[2])

    class_counts = Counter()
    for _, filename, _ in successful:
        label_path = (
            OUT / "labels" / short / (Path(filename).stem + ".txt")
        )
        for line in label_path.read_text().splitlines():
            class_counts[int(line.split()[0])] += 1

    print("Valid instances per class:")
    for class_id, name in enumerate(CLASSES):
        print(f"  {name:10s}: {class_counts[class_id]}")

for split in ["train2017", "val2017"]:
    prepare_split(split)

(OUT / "data.yaml").write_text(
    "path: .\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n" +
    "".join(f"  {i}: {name}\n" for i, name in enumerate(CLASSES))
)

print("\nDataset created at:", OUT)
print("Remember: this cell prepares data only; it does not train a model.")


Preparing train: 500 selected images
Successful images: 500
Failed images: 0
Valid instances per class:
  keyboard  : 500
  mouse     : 424
  bottle    : 735
  cup       : 684
  person    : 1031

Preparing val: 100 selected images
Successful images: 100
Failed images: 0
Valid instances per class:
  keyboard  : 85
  mouse     : 60
  bottle    : 197
  cup       : 183
  person    : 256

Dataset created at: /content/rk3576_yolo_dataset/coco5_yolo
Remember: this cell prepares data only; it does not train a model.


In [ ]:

from pathlib import Path

OUT = Path("/content/rk3576_yolo_dataset/coco5_yolo")

for split in ["train", "val"]:
    images = list((OUT / "images" / split).glob("*.jpg"))
    labels = list((OUT / "labels" / split).glob("*.txt"))

    print(f"\n{split}:")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Matching counts:", len(images) == len(labels))

    if images:
        sample = images[0]
        label = OUT / "labels" / split / (sample.stem + ".txt")
        print("Sample image:", sample.name)
        print("Sample labels:")
        print(label.read_text()[:1000])



train:
Images: 500
Labels: 500
Matching counts: True
Sample image: 000000577160.jpg
Sample labels:
2 0.279779 0.242132 0.231454 0.478644
2 0.478497 0.216560 0.208595 0.425801
4 0.829216 0.100417 0.341569 0.200833
3 0.190670 0.421609 0.298301 0.197827


val:
Images: 100
Labels: 100
Matching counts: True
Sample image: 000000127394.jpg
Sample labels:
2 0.830672 0.274016 0.032103 0.027531
4 0.112702 0.329734 0.225403 0.389562
3 0.626626 0.285750 0.058240 0.049781
3 0.918778 0.524758 0.106455 0.095672
3 0.869487 0.296602 0.060538 0.080109
3 0.428203 0.344781 0.058998 0.065031
4 0.320452 0.254641 0.312200 0.267500
4 0.545171 0.215906 0.264866 0.269656
2 0.798582 0.294375 0.035795 0.091094
2 0.766064 0.307984 0.049389 0.068688
4 0.903203 0.425641 0.193594 0.139344
4 0.979132 0.186570 0.041736 0.084672
4 0.716235 0.158078 0.160978 0.163937



In [ ]:
from pathlib import Path
import shutil
import os

OUT = Path("/content/rk3576_yolo_dataset/coco5_yolo")
ZIP_BASE = "/content/rk3576_yolo_dataset/rk3576_coco5_dataset"

assert (OUT / "images/train").is_dir(), "Training images folder missing"
assert (OUT / "images/val").is_dir(), "Validation images folder missing"
assert (OUT / "labels/train").is_dir(), "Training labels folder missing"
assert (OUT / "labels/val").is_dir(), "Validation labels folder missing"
assert (OUT / "data.yaml").is_file(), "data.yaml missing"

zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=OUT
)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print("Dataset ZIP created successfully")
print(f"Path: {zip_path}")
print(f"Size: {size_mb:.2f} MB")

Dataset ZIP created successfully
Path: /content/rk3576_yolo_dataset/rk3576_coco5_dataset.zip
Size: 3.53 MB


In [ ]:
from google.colab import files

files.download(
    "/content/rk3576_yolo_dataset/rk3576_coco5_dataset.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>